In [ ]:
# ----------------------------
# Setup the Jupyter version of Dash
# ----------------------------
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
import urllib.parse
import os

# ----------------------------
# CRUD Module import
# ----------------------------
from CRUD_Python_Module import AnimalShelter


# ----------------------------
# RESCUE CRITERIA CONFIGURATION
# Enhancement: extracted from the callback into a top-level config dictionary.
# Changing criteria now requires editing this block only — not the callback logic.
# Breed lists and age ranges come from Grazioso Salvare's training requirements.
# ----------------------------
RESCUE_CRITERIA = {
    # Water Rescue: Labrador/Chesapeake/Newfoundland breeds are strong swimmers;
    # Intact Female preferred; age cap of 26 weeks targets optimal trainability window.
    "WR": {
        "animal_type": "Dog",
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female",
        "age_max_weeks": 26,
    },
    # Mountain/Wilderness Rescue: Working/Nordic breeds suited for cold terrain;
    # Intact Male preferred; same 26-week training window.
    "MWR": {
        "animal_type": "Dog",
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky Mix"],
        "sex": "Intact Male",
        "age_max_weeks": 26,
    },
    # Disaster/Individual Tracking: Scent-tracking breeds; Intact Male;
    # wider age range (20-300 weeks) allows more experienced dogs.
    "DIT": {
        "animal_type": "Dog",
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound"],
        "sex": "Intact Male",
        "age_min_weeks": 20,
        "age_max_weeks": 300,
    },
}


# ----------------------------
# Data / Model
# ----------------------------

# Enhancement: credentials loaded from environment variables instead of hardcoded.
# Set AAC_USERNAME and AAC_PASSWORD in your shell or .env before running.
raw_username = os.environ.get("AAC_USERNAME")
raw_password = os.environ.get("AAC_PASSWORD")

if not raw_username or not raw_password:
    raise EnvironmentError(
        "Missing required environment variables: AAC_USERNAME and/or AAC_PASSWORD. "
        "Set them before running this notebook (e.g., export AAC_USERNAME=aacuser)."
    )

username = urllib.parse.quote_plus(raw_username)
password = urllib.parse.quote_plus(raw_password)

shelter = AnimalShelter(username, password)


def build_query_from_criteria(criteria: dict) -> dict:
    """Translate a RESCUE_CRITERIA entry into a MongoDB query document."""
    query = {
        "animal_type": criteria["animal_type"],
        "breed": {"$in": criteria["breeds"]},
        "sex_upon_outcome": criteria["sex"],
    }
    # Build age range filter only from whichever bounds are present
    age_filter = {}
    if "age_min_weeks" in criteria:
        age_filter["$gte"] = criteria["age_min_weeks"]
    if "age_max_weeks" in criteria:
        age_filter["$lte"] = criteria["age_max_weeks"]
    if age_filter:
        query["age_upon_outcome_in_weeks"] = age_filter
    return query


def fetch_df(query: dict) -> pd.DataFrame:
    """Fetch records from Mongo via the CRUD module and return a cleaned DataFrame."""
    try:
        records = shelter.read(query)
        dff = pd.DataFrame.from_records(records)
        if not dff.empty and "_id" in dff.columns:
            dff.drop(columns=["_id"], inplace=True)
        return dff
    except Exception as e:
        print(f"[fetch_df] Error reading documents: {e}")
        return pd.DataFrame()


# Initial load — all records used for the startup table render
df = fetch_df({})
print("Number of records read from MongoDB:", len(df))


# ----------------------------
# Helpers
# ----------------------------
def guess_lat_lon_columns(columns):
    """Try to detect latitude and longitude column names by matching common aliases."""
    cols = [c.lower() for c in columns]
    lat_candidates = ["location_lat", "lat", "latitude", "geo_lat", "y"]
    lon_candidates = ["location_long", "location_lng", "lon", "lng", "longitude", "geo_long", "geo_lng", "x"]

    lat_col = None
    lon_col = None

    for cand in lat_candidates:
        if cand in cols:
            lat_col = columns[cols.index(cand)]
            break

    for cand in lon_candidates:
        if cand in cols:
            lon_col = columns[cols.index(cand)]
            break

    return lat_col, lon_col


def build_logo_img(filename: str):
    """
    Robust logo loader:
    - asserts file exists (so you see errors immediately)
    - detects PNG vs JPEG by header bytes
    - returns an html.Img that is forced visible
    """
    assert os.path.exists(filename), f"Logo not found in CWD: {os.getcwd()} -> {filename}"

    with open(filename, "rb") as f:
        data = f.read()

    # PNG signature: 89 50 4E 47 0D 0A 1A 0A
    # JPEG signature: FF D8 FF
    if data[:8] == b"\x89PNG\r\n\x1a\n":
        mime = "image/png"
    elif data[:3] == b"\xff\xd8\xff":
        mime = "image/jpeg"
    else:
        mime = "image/png"

    encoded = base64.b64encode(data).decode("ascii")

    return html.Img(
        src=f"data:{mime};base64,{encoded}",
        style={
            "height": "110px",
            "display": "block",
            "margin": "0 auto",
            "border": "1px solid #ccc",
            "padding": "4px",
            "background": "white"
        },
    )


# ----------------------------
# Dash App / Layout
# ----------------------------
app = JupyterDash(__name__)

logo_img = build_logo_img("Grazioso Salvare Logo.png")

app.layout = html.Div([
    html.Div([
        logo_img,
        html.Center(html.B(html.H1("SNHU CS-340 Dashboard"))),
        html.Center(html.P("Amanda Willbanks")),
    ]),

    html.Hr(),

    # ----------------------------
    # FILTER UI
    # ----------------------------
    html.Div([
        html.H4("Filter Type"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "All animals", "value": "ALL"},
                {"label": "Water Rescue", "value": "WR"},
                {"label": "Mountain/Wilderness Rescue", "value": "MWR"},
                {"label": "Disaster/Individual Tracking", "value": "DIT"},
            ],
            value="ALL",
            labelStyle={"display": "block"},
        ),
    ], style={"padding": "10px"}),

    html.Hr(),

    # ----------------------------
    # DATATABLE
    # selected_rows=[0] ensures the first row is always selected on load
    # so the map and chart have data to render immediately.
    # ----------------------------
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),

        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        column_selectable="single",

        style_table={"height": "400px", "overflowY": "auto", "overflowX": "auto"},
        style_cell={"textAlign": "left", "fontFamily": "Arial", "fontSize": 12, "minWidth": "120px"},
    ),

    html.Br(),
    html.Hr(),

    # Side-by-side Graph + Map
    html.Div(
        className="row",
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", className="col s12 m6", style={"flex": "1"}),
            html.Div(id="map-id", className="col s12 m6", style={"flex": "1"}),
        ],
    ),
])


# ----------------------------
# Callbacks
# ----------------------------

# 1) Filter -> update DataTable from MongoDB
@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "columns"),
    Input("filter-type", "value"),
)
def update_dashboard(filter_type):
    # Enhancement: RESCUE_CRITERIA dictionary drives query construction.
    # Adding a new rescue type requires only a new entry in RESCUE_CRITERIA above.
    if filter_type == "ALL":
        query = {}
    elif filter_type in RESCUE_CRITERIA:
        query = build_query_from_criteria(RESCUE_CRITERIA[filter_type])
    else:
        # Raise explicitly so unexpected values surface as visible errors
        # rather than silently returning all records.
        raise ValueError(f"Unknown filter type: {filter_type!r}")

    dff = fetch_df(query)
    if dff.empty:
        return [], [{"name": i, "id": i} for i in df.columns]

    return dff.to_dict("records"), [{"name": i, "id": i} for i in dff.columns]


# 2) Highlight selected column
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def update_styles(selected_columns):
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in (selected_columns or [])]


# 3) Graph based on visible table data
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_graphs(viewData):
    if not viewData:
        return html.Div("No data to graph.")

    dff = pd.DataFrame(viewData)

    group_col = "breed" if "breed" in dff.columns else dff.columns[0]

    counts = dff[group_col].value_counts().head(10).reset_index()
    counts.columns = [group_col, "count"]

    fig = px.pie(counts, names=group_col, values="count", title=f"Top {group_col} (visible rows)")
    return dcc.Graph(figure=fig)


# 4) Map update
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows"),
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    row = 0 if not index else index[0]

    if dff.empty or row < 0 or row >= len(dff):
        return [html.Div("No data available to display on the map.")]

    lat_col, lon_col = guess_lat_lon_columns(dff.columns)

    try:
        if lat_col and lon_col:
            lat = float(dff.loc[row, lat_col])
            lon = float(dff.loc[row, lon_col])
        else:
            # Fallback to positional index only when named columns cannot be detected
            lat = float(dff.iloc[row, 13])
            lon = float(dff.iloc[row, 14])
    except Exception:
        return [html.Div("Could not determine valid latitude/longitude for selected row.")]

    # Enhancement: use named column lookups instead of magic integer indexes.
    # 'breed' is used as the tooltip (what the user hovers over on the map marker).
    # 'name' is the animal's name shown in the popup.
    TOOLTIP_COL = "breed"
    NAME_COL = "name"

    if TOOLTIP_COL in dff.columns:
        tooltip_text = str(dff.loc[row, TOOLTIP_COL])
    else:
        tooltip_text = "Animal"

    if NAME_COL in dff.columns:
        animal_name = str(dff.loc[row, NAME_COL])
    else:
        animal_name = "(no name)"

    # Map is centered on Austin, TX (30.75, -97.48) — the location of the
    # Austin Animal Center, which is the source of this dataset.
    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(tooltip_text),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(animal_name)
                        ])
                    ],
                ),
            ],
        )
    ]


# ----------------------------
# Run
# ----------------------------
app.run_server(debug=True, port=8052)
